In [11]:
import pandas as pd
import re
import openpyxl

file_path = '../PM Daily Report/PM 17/DAILY REPORT PM 17 APRIL 2026.xlsx'
df = pd.read_excel(file_path, sheet_name='BB', header=None, engine='openpyxl')

patterns = {
    "Date": r"DATE",
    "Grade": r"GRADE",
    "GSM": r"GSM",
    "Total NBKP": r"TOTAL NBKP",
    "Total LBKP": r"TOTAL LBKP",
    "Total BB Recycle": r"TOTAL BB Recycle",
    "Sub Total Pulp+Broke": r"SUB TOTAL PULP\+BROKE"
}

row_8 = df.iloc[7]
column_mapping = {}

# 1. Cari indeks kolom utama menggunakan Regex
for label, pola in patterns.items():
    for col_idx, isi_sel in row_8.items():
        if re.search(pola, str(isi_sel), re.IGNORECASE):
            column_mapping[label] = col_idx
            break

# 2. Definisikan relasi spasial untuk kolom persentase (+1 index)
relasi_persentase = {
    "% NBKP": "Total NBKP",
    "% LBKP": "Total LBKP",
    "% BB Recycle": "Total BB Recycle",
    "% Pulp+Broke": "Sub Total Pulp+Broke"
}

# Injeksi langsung nilai indeksnya tanpa perlu scanning (lebih cepat dan aman)
for label_persen, label_base in relasi_persentase.items():
    if label_base in column_mapping:
        column_mapping[label_persen] = column_mapping[label_base] + 1

# 3. Ekstraksi Data (Rentang disesuaikan dengan file .ipynb Anda: 10-40)
start_row, end_row = 9, 40
extracted_data = {}

for label, col_idx in column_mapping.items():
    data_series = df.iloc[start_row:end_row, col_idx].copy()
    extracted_data[label] = data_series.values

result_df = pd.DataFrame(extracted_data)

# 4. Pembersihan Data Dinamis
result_df["Date"] = result_df["Date"].fillna(0)
result_df["Grade"] = result_df["Grade"].replace('-', 0).fillna(0)
result_df["GSM"] = result_df["GSM"].replace('-', 0).fillna(0)

# Alih-alih menulis ulang nama kolom untuk dikonversi ke numerik, 
# kita isolasi semua kolom selain Date, Grade, dan GSM agar diubah menjadi angka.
kolom_teks = ["Date", "Grade", "GSM"]
for col in result_df.columns:
    if col not in kolom_teks:
        result_df[col] = result_df[col].replace('-', 0).fillna(0)
        result_df[col] = pd.to_numeric(result_df[col], errors='coerce').fillna(0)
result_df.head()

result_df.to_excel('BB_April_PM17.xlsx', index=False)